In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

In [2]:
import yaml
try:
    with open("../config.yaml", "r") as file:
        cfg=yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

# 1. NVIDIA

**IMPORTANTE**: año fiscal de NVIDIA termina en enero asi que trimestres no son al uso: marzo, junio, sept, dic. Va a ser: Q1-abril, Q2-julio, Q3-octubre, Q4-enero.

In [3]:
import requests

In [4]:
import pandas as pd
import requests

HEADERS = {"User-Agent": "irenesanch.medina@gmail.com"}
CIK = "0001045810"

facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"
data = requests.get(facts_url, headers=HEADERS).json()
gaap = data["facts"]["us-gaap"]


def concept_facts(gaap, concept, unit="USD", start="2021-01-01"):
    rows = []

    for e in gaap[concept]["units"].get(unit, []):
        if e.get("form") not in ["10-Q", "10-K"]:
            continue

        end = pd.to_datetime(e["end"])
        if end < pd.to_datetime(start):
            continue

        start_date = pd.to_datetime(e.get("start", pd.NaT))
        days = None if pd.isna(start_date) else (end - start_date).days

        rows.append({
            "fy": e.get("fy"),
            "fp": e.get("fp"),
            "form": e.get("form"),
            "filed": pd.to_datetime(e.get("filed")),
            "start": start_date,
            "end": end,
            "days": days,
            "val": e["val"],})

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    # drop duplicate SEC contexts keep only last filed value
    df = (df.sort_values(["fy", "fp", "end", "days", "filed"]).drop_duplicates(["fy", "fp", "end", "days"], keep="last"))

    return df


def get_balance_sheet_quarterly(gaap, concept, start="2021-01-01"):
    raw = concept_facts(gaap, concept, start=start)
    if raw.empty:
        return pd.Series(dtype=float, name=concept)

    # balance sheet facts are moemtn in time no duration filter
    out = (raw.sort_values(["end", "filed"]).drop_duplicates("end", keep="last").set_index("end")["val"].sort_index())

    out.name = concept
    return out


def get_flow_quarterly(gaap, concept, start="2021-01-01"):
    raw = concept_facts(gaap, concept, start=start)
    if raw.empty:
        return pd.Series(dtype=float, name=concept)

    records = []

    for fy, g in raw.groupby("fy"):
        g = g.sort_values(["end", "days", "filed"])

        q1 = g[(g["fp"] == "Q1") & (g["days"].between(80, 100))]
        q2_ytd = g[(g["fp"] == "Q2") & (g["days"].between(170, 195))]
        q2_q = g[(g["fp"] == "Q2") & (g["days"].between(80, 100))]
        q3_ytd = g[(g["fp"] == "Q3") & (g["days"].between(260, 285))]
        q3_q = g[(g["fp"] == "Q3") & (g["days"].between(80, 100))]
        fy_full = g[(g["fp"] == "FY") & (g["days"].between(350, 380))]
        q4_q = g[(g["fp"] == "FY") & (g["days"].between(80, 100))]

        q1_val = q1.iloc[-1] if not q1.empty else None
        q2_ytd_val = q2_ytd.iloc[-1] if not q2_ytd.empty else None
        q2_q_val = q2_q.iloc[-1] if not q2_q.empty else None
        q3_ytd_val = q3_ytd.iloc[-1] if not q3_ytd.empty else None
        q3_q_val = q3_q.iloc[-1] if not q3_q.empty else None
        fy_val = fy_full.iloc[-1] if not fy_full.empty else None
        q4_q_val = q4_q.iloc[-1] if not q4_q.empty else None

        if q1_val is not None:
            records.append({"quarter_end": q1_val["end"], "value": q1_val["val"]})

        if q2_q_val is not None:
            records.append({"quarter_end": q2_q_val["end"], "value": q2_q_val["val"]})
        elif q1_val is not None and q2_ytd_val is not None:
            records.append({
                "quarter_end": q2_ytd_val["end"],
                "value": q2_ytd_val["val"] - q1_val["val"],})

        if q3_q_val is not None:
            records.append({"quarter_end": q3_q_val["end"], "value": q3_q_val["val"]})
        elif q2_ytd_val is not None and q3_ytd_val is not None:
            records.append({
                "quarter_end": q3_ytd_val["end"],
                "value": q3_ytd_val["val"] - q2_ytd_val["val"],})

        if q4_q_val is not None:
            records.append({"quarter_end": q4_q_val["end"], "value": q4_q_val["val"]})
        elif q3_ytd_val is not None and fy_val is not None:
            records.append({
                "quarter_end": fy_val["end"],
                "value": fy_val["val"] - q3_ytd_val["val"],})

    if not records:
        return pd.Series(dtype=float, name=concept)

    out = (pd.DataFrame(records).sort_values("quarter_end").drop_duplicates("quarter_end", keep="last").set_index("quarter_end")["value"].sort_index())

    out.name = concept
    return out

In [5]:
df = pd.DataFrame()

# flow statement items
df["revenue_q"]= get_flow_quarterly(gaap, "RevenueFromContractWithCustomerExcludingAssessedTax")
if df["revenue_q"].dropna().empty:
    df["revenue_q"]= get_flow_quarterly(gaap, "Revenues")

df["gross_profit_q"]= get_flow_quarterly(gaap, "GrossProfit")
df["ebit_q"]= get_flow_quarterly(gaap, "OperatingIncomeLoss")
df["net_income_q"]= get_flow_quarterly(gaap, "NetIncomeLoss")
df["rd_expense_q"]= get_flow_quarterly(gaap, "ResearchAndDevelopmentExpense")

# cash flow items: need YTD differencing
df["capex_q"] = get_flow_quarterly(gaap, "PaymentsToAcquireProductiveAssets")
df["operating_cf_q"] = get_flow_quarterly(gaap, "NetCashProvidedByUsedInOperatingActivities")

# balance sheet items
df["cash_q"] = get_balance_sheet_quarterly(gaap, "CashAndCashEquivalentsAtCarryingValue")
df["total_equity_q"] = get_balance_sheet_quarterly(gaap, "StockholdersEquity")

lt_debt = get_balance_sheet_quarterly(gaap, "LongTermDebt")
st_debt = get_balance_sheet_quarterly(gaap, "DebtCurrent")
df["total_debt_q"] = lt_debt.add(st_debt, fill_value=0)

df["ticker"] = "NVDA"
df.index.name = "quarter_end"

df = df[df.index >= "2021-01-01"].sort_index()

print(df.shape)
print(df.isna().sum())
print(df.tail())

(21, 11)
revenue_q         0
gross_profit_q    0
ebit_q            0
net_income_q      0
rd_expense_q      0
capex_q           8
operating_cf_q    0
cash_q            0
total_equity_q    0
total_debt_q      0
ticker            0
dtype: int64
               revenue_q  gross_profit_q       ebit_q  net_income_q  \
quarter_end                                                           
2025-04-27   44062000000     26668000000  21638000000   18775000000   
2025-07-27   46743000000     33853000000  28440000000   26422000000   
2025-10-26   57006000000     41849000000  36010000000   31910000000   
2026-01-25   68127000000     51093000000  44299000000   42960000000   
2026-04-26   81615000000     61157000000  53536000000   58321000000   

             rd_expense_q       capex_q  operating_cf_q       cash_q  \
quarter_end                                                            
2025-04-27     3989000000  1.227000e+09     27414000000  15234000000   
2025-07-27     4291000000  1.895000e+09     

In [6]:
# checking nulls for capex
print("Capex coverage:")
print(df[["capex_q"]].to_string())

Capex coverage:
                  capex_q
quarter_end              
2021-05-02   2.980000e+08
2021-08-01   1.830000e+08
2021-10-31   2.220000e+08
2022-01-30            NaN
2022-05-01            NaN
2022-07-31            NaN
2022-10-30            NaN
2023-01-29            NaN
2023-04-30            NaN
2023-07-30            NaN
2023-10-29            NaN
2024-01-28   2.540000e+08
2024-04-28   3.690000e+08
2024-07-28   9.770000e+08
2024-10-27   8.130000e+08
2025-01-26   1.077000e+09
2025-04-27   1.227000e+09
2025-07-27   1.895000e+09
2025-10-26   1.636000e+09
2026-01-25   1.284000e+09
2026-04-26   1.757000e+09


In [7]:
# check PaymentsToAcquireProductiveAssets raw entries to see why the missing data around 2022-2023
print("PaymentsToAcquireProductiveAssets")
for e in gaap["PaymentsToAcquireProductiveAssets"]["units"]["USD"]:
    start = pd.to_datetime(e.get("start", pd.NaT))
    end= pd.to_datetime(e["end"])
    days= (end - start).days if pd.notna(start) else None
    if end.year >= 2021:
        print(f"{e.get('form')} | {e.get('fp')} | {e.get('start')} → {e['end']} | {e['val']:,} | {days} days")

PaymentsToAcquireProductiveAssets
10-Q | Q1 | 2021-02-01 → 2021-05-02 | 298,000,000 | 90 days
10-Q | Q2 | 2021-02-01 → 2021-08-01 | 481,000,000 | 181 days
10-Q | Q3 | 2021-02-01 → 2021-10-31 | 703,000,000 | 272 days
10-K | FY | 2021-02-01 → 2022-01-30 | 976,000,000 | 363 days
10-Q | Q3 | 2022-01-31 → 2022-10-30 | 1,324,000,000 | 272 days
10-K | FY | 2022-01-31 → 2023-01-29 | 1,833,000,000 | 363 days
10-K | FY | 2022-01-31 → 2023-01-29 | 1,833,000,000 | 363 days
10-Q | Q1 | 2023-01-30 → 2023-04-30 | 248,000,000 | 90 days
10-Q | Q2 | 2023-01-30 → 2023-07-30 | 537,000,000 | 181 days
10-Q | Q3 | 2023-01-30 → 2023-10-29 | 815,000,000 | 272 days
10-Q | Q3 | 2023-01-30 → 2023-10-29 | 815,000,000 | 272 days
10-K | FY | 2023-01-30 → 2024-01-28 | 1,069,000,000 | 363 days
10-K | FY | 2023-01-30 → 2024-01-28 | 1,069,000,000 | 363 days
10-K | FY | 2023-01-30 → 2024-01-28 | 1,069,000,000 | 363 days
10-Q | Q1 | 2024-01-29 → 2024-04-28 | 369,000,000 | 90 days
10-Q | Q1 | 2024-01-29 → 2024-04-28 | 369,

In [8]:
# the values are there just filed anually for that period not quarterly, calculate them manually 
def get_capex_nvda_v2(gaap, concept="PaymentsToAcquireProductiveAssets", start="2021-01-01"):
    entries = gaap[concept]["units"]["USD"]
    
    records = []
    for e in entries:
        if e.get("form") not in ["10-Q", "10-K"]:
            continue
        end= pd.to_datetime(e["end"])
        start_date = pd.to_datetime(e.get("start", pd.NaT))
        if end < pd.to_datetime(start) or pd.isna(start_date):
            continue
        days = (end - start_date).days
        records.append({
            "fp":e.get("fp"),
            "end":end,
            "start":start_date,
            "days":days,
            "val":e["val"]})
    
    df = (pd.DataFrame(records).sort_values(["start", "end", "days"]).drop_duplicates(["start", "fp"], keep="last"))  # remove duplicates by start+fp
    
    results = {}
    
    # group by fiscal year start date (more reliable than fy field thata is duplicated)
    for fy_start, g in df.groupby("start"):
        g = g.sort_values("end")
        
        q1= g[(g["fp"] == "Q1") & g["days"].between(80,  100)]
        q2= g[(g["fp"] == "Q2") & g["days"].between(170, 195)]
        q3= g[(g["fp"] == "Q3") & g["days"].between(260, 285)]
        fy_= g[(g["fp"] == "FY") & g["days"].between(350, 380)]
        
        q1_val= q1.iloc[-1]["val"]  if not q1.empty  else None
        q2_val= q2.iloc[-1]["val"]  if not q2.empty  else None
        q3_val= q3.iloc[-1]["val"]  if not q3.empty  else None
        fy_val= fy_.iloc[-1]["val"] if not fy_.empty else None
        q1_end= q1.iloc[-1]["end"]  if not q1.empty  else None
        q2_end= q2.iloc[-1]["end"]  if not q2.empty  else None
        q3_end= q3.iloc[-1]["end"]  if not q3.empty  else None
        fy_end= fy_.iloc[-1]["end"] if not fy_.empty else None

        # Q1: always standalone
        if q1_val is not None:
            results[q1_end] = q1_val

        # Q2 = Q2_YTD - Q1
        if q2_val is not None and q1_val is not None:
            results[q2_end] = q2_val - q1_val

        # Q3 = Q3_YTD - Q2_YTD
        if q3_val is not None and q2_val is not None:
            results[q3_end] = q3_val - q2_val
        elif q3_val is not None and q1_val is not None:
            # fallback if no Q2: Q3_YTD - Q1 (less accurate, covers FY2022 case)
            results[q3_end] = q3_val - q1_val

        # Q4 = FY - Q3_YTD
        if fy_val is not None and q3_val is not None:
            results[fy_end] = fy_val - q3_val
        # FY2022 special case: only Q3_YTD and FY, no Q1/Q2
        elif fy_val is not None and q3_val is not None and q1_val is None:
            results[fy_end] = fy_val - q3_val

    return pd.Series(results, name="capex_q").sort_index()

capex_v2 = get_capex_nvda_v2(gaap)
print(f"Entries: {capex_v2.notna().sum()}")
print(capex_v2)

Entries: 18
2021-05-02     298000000
2021-08-01     183000000
2021-10-31     222000000
2022-01-30     273000000
2023-01-29     509000000
2023-04-30     248000000
2023-07-30     289000000
2023-10-29     278000000
2024-01-28     254000000
2024-04-28     369000000
2024-07-28     977000000
2024-10-27     813000000
2025-01-26    1077000000
2025-04-27    1227000000
2025-07-27    1895000000
2025-10-26    1636000000
2026-01-25    1284000000
2026-04-26    1757000000
Name: capex_q, dtype: int64


In [9]:
# 2022 missing quarters: May, July, October 2022
# Q1+Q2+Q3 total = Q3_YTD - Q4_prev = 1,324M - 273M = 1,051M
fy2022_q123_total = 1_324_000_000 - 273_000_000
per_quarter = fy2022_q123_total / 3

df.loc["2022-05-01", "capex_q"] = per_quarter
df.loc["2022-07-31", "capex_q"] = per_quarter
df.loc["2022-10-30", "capex_q"] = per_quarter

print("Capex NaNs after fill:", df["capex_q"].isna().sum())
print(df["capex_q"].to_string())

Capex NaNs after fill: 5
quarter_end
2021-05-02    2.980000e+08
2021-08-01    1.830000e+08
2021-10-31    2.220000e+08
2022-01-30             NaN
2022-05-01    3.503333e+08
2022-07-31    3.503333e+08
2022-10-30    3.503333e+08
2023-01-29             NaN
2023-04-30             NaN
2023-07-30             NaN
2023-10-29             NaN
2024-01-28    2.540000e+08
2024-04-28    3.690000e+08
2024-07-28    9.770000e+08
2024-10-27    8.130000e+08
2025-01-26    1.077000e+09
2025-04-27    1.227000e+09
2025-07-27    1.895000e+09
2025-10-26    1.636000e+09
2026-01-25    1.284000e+09
2026-04-26    1.757000e+09


In [10]:
# assign capex_v2 directly to model_df by reindexing
df["capex_q"] = capex_v2.reindex(df.index)

# fill FY2022 Q1/Q2/Q3
fy2022_q123 = (1_324_000_000 - 273_000_000) / 3
df.loc["2022-05-01", "capex_q"] = fy2022_q123
df.loc["2022-07-31", "capex_q"] = fy2022_q123
df.loc["2022-10-30", "capex_q"] = fy2022_q123

print("Capex NaNs:", df["capex_q"].isna().sum())
print(df["capex_q"])

Capex NaNs: 0
quarter_end
2021-05-02    2.980000e+08
2021-08-01    1.830000e+08
2021-10-31    2.220000e+08
2022-01-30    2.730000e+08
2022-05-01    3.503333e+08
2022-07-31    3.503333e+08
2022-10-30    3.503333e+08
2023-01-29    5.090000e+08
2023-04-30    2.480000e+08
2023-07-30    2.890000e+08
2023-10-29    2.780000e+08
2024-01-28    2.540000e+08
2024-04-28    3.690000e+08
2024-07-28    9.770000e+08
2024-10-27    8.130000e+08
2025-01-26    1.077000e+09
2025-04-27    1.227000e+09
2025-07-27    1.895000e+09
2025-10-26    1.636000e+09
2026-01-25    1.284000e+09
2026-04-26    1.757000e+09
Name: capex_q, dtype: float64


In [11]:
# find depreciation and amortization tag to calculate EBITDA manully:
print("D&A concepts with quarterly data:")
for concept, details in gaap.items():
    if any(word in concept.lower() for word in ["depreciation", "amortization"]):
        try:
            entries = details["units"]["USD"]
            quarterly_90 = [e for e in entries 
                           if e.get("form") in ["10-Q", "10-K"]
                           and 80 <= (pd.to_datetime(e["end"]) - 
                                     pd.to_datetime(e.get("start", e["end"]))).days <= 100]
            if quarterly_90:
                dates = [pd.to_datetime(e["end"]).year for e in quarterly_90]
                print(f"  {concept}: {len(quarterly_90)} entries, years: {min(dates)}-{max(dates)}")
        except:
            continue

D&A concepts with quarterly data:
  AmortizationOfAcquiredIntangibleAssets: 8 entries, years: 2012-2014
  AmortizationOfDebtDiscountPremium: 18 entries, years: 2013-2017
  AmortizationOfFinancingCosts: 5 entries, years: 2013-2014
  AmortizationOfFinancingCostsAndDiscounts: 8 entries, years: 2015-2017
  AmortizationOfIntangibleAssets: 80 entries, years: 2012-2026
  CostOfGoodsSoldAmortization: 3 entries, years: 2011-2011
  DepreciationAndAmortization: 64 entries, years: 2009-2021
  DepreciationDepletionAndAmortization: 10 entries, years: 2021-2026
  FiniteLivedIntangibleAssetsAmortizationExpense: 16 entries, years: 2009-2012
  FutureAmortizationExpenseAfterYearFive: 1 entries, years: 2012-2012
  FutureAmortizationExpenseRemainderOfFiscalYear: 1 entries, years: 2012-2012
  FutureAmortizationExpenseYearFour: 2 entries, years: 2011-2012
  FutureAmortizationExpenseYearOne: 2 entries, years: 2011-2012
  FutureAmortizationExpenseYearThree: 2 entries, years: 2011-2012
  FutureAmortizationExpen

In [12]:
# use new tag where available, fall back to old tag when missing
da_old = get_flow_quarterly(gaap, "DepreciationAndAmortization")
da_new = get_flow_quarterly(gaap, "DepreciationDepletionAndAmortization")

da_combined = da_new.combine_first(da_old)

print("Combined D&A coverage:")
print(f"non-null entries: {da_combined.notna().sum()}")
print(f"years: {da_combined.index.min()} to {da_combined.index.max()}")
print(da_combined)

Combined D&A coverage:
non-null entries: 20
years: 2021-05-02 00:00:00 to 2026-04-26 00:00:00
quarter_end
2021-05-02    281000000
2021-08-01    286000000
2022-01-30    309000000
2022-05-01    334000000
2022-07-31    378000000
2022-10-30    406000000
2023-01-29    426000000
2023-04-30    384000000
2023-07-30    365000000
2023-10-29    372000000
2024-01-28    387000000
2024-04-28    410000000
2024-07-28    433000000
2024-10-27    478000000
2025-01-26    543000000
2025-04-27    611000000
2025-07-27    669000000
2025-10-26    751000000
2026-01-25    812000000
2026-04-26    997000000
Name: DepreciationDepletionAndAmortization, dtype: int64


In [13]:
# assign D&A to df
df["da_q"] = da_combined.reindex(df.index)

print("D&A NaNs:", df["da_q"].isna().sum())
print(df["da_q"])

D&A NaNs: 1
quarter_end
2021-05-02    281000000.0
2021-08-01    286000000.0
2021-10-31            NaN
2022-01-30    309000000.0
2022-05-01    334000000.0
2022-07-31    378000000.0
2022-10-30    406000000.0
2023-01-29    426000000.0
2023-04-30    384000000.0
2023-07-30    365000000.0
2023-10-29    372000000.0
2024-01-28    387000000.0
2024-04-28    410000000.0
2024-07-28    433000000.0
2024-10-27    478000000.0
2025-01-26    543000000.0
2025-04-27    611000000.0
2025-07-27    669000000.0
2025-10-26    751000000.0
2026-01-25    812000000.0
2026-04-26    997000000.0
Name: da_q, dtype: float64


In [14]:
# interpolate missing depreciation value for 2021-10-31
df["da_q"] = df["da_q"].interpolate(method="linear")

print(df["da_q"])

quarter_end
2021-05-02    281000000.0
2021-08-01    286000000.0
2021-10-31    297500000.0
2022-01-30    309000000.0
2022-05-01    334000000.0
2022-07-31    378000000.0
2022-10-30    406000000.0
2023-01-29    426000000.0
2023-04-30    384000000.0
2023-07-30    365000000.0
2023-10-29    372000000.0
2024-01-28    387000000.0
2024-04-28    410000000.0
2024-07-28    433000000.0
2024-10-27    478000000.0
2025-01-26    543000000.0
2025-04-27    611000000.0
2025-07-27    669000000.0
2025-10-26    751000000.0
2026-01-25    812000000.0
2026-04-26    997000000.0
Name: da_q, dtype: float64


In [15]:
# calculate EBITA now that we have D&A
df["ebitda_q"] = df["ebit_q"] + df["da_q"]

In [16]:
df.head(20)

,revenue_q,gross_profit_q,ebit_q,net_income_q,rd_expense_q,capex_q,operating_cf_q,cash_q,total_equity_q,total_debt_q,ticker,da_q,ebitda_q
quarter_end,,,,,,,,,,,,,
2021-05-02,5661000000,3629000000,1956000000,1912000000,1153000000,2.980000e+08,1874000000,978000000,18774000000,7.962000e+09,NVDA,281000000.0,2.237000e+09
2021-08-01,6507000000,4215000000,2444000000,2374000000,1245000000,1.830000e+08,2682000000,5628000000,21147000000,1.294300e+10,NVDA,286000000.0,2.730000e+09
2021-10-31,7103000000,4631000000,2671000000,2464000000,1403000000,2.220000e+08,1519000000,1288000000,23798000000,1.094400e+10,NVDA,297500000.0,2.968500e+09
2022-01-30,7643000000,4999000000,2970000000,3003000000,1466000000,2.730000e+08,3033000000,1990000000,26612000000,1.094600e+10,NVDA,309000000.0,3.279000e+09
2022-05-01,8288000000,5431000000,1868000000,1618000000,1618000000,3.503333e+08,1731000000,3887000000,26320000000,1.094700e+10,NVDA,334000000.0,2.202000e+09
2022-07-31,6704000000,2915000000,499000000,656000000,1824000000,3.503333e+08,1270000000,3013000000,23851000000,1.219800e+10,NVDA,378000000.0,8.770000e+08
2022-10-30,5931000000,3177000000,601000000,680000000,1945000000,3.503333e+08,392000000,2800000000,21349000000,1.219900e+10,NVDA,406000000.0,1.007000e+09
2023-01-29,6051000000,3833000000,1256000000,1414000000,1952000000,5.090000e+08,2248000000,3389000000,22101000000,1.220300e+10,NVDA,426000000.0,1.682000e+09
2023-04-30,7192000000,4648000000,2140000000,2043000000,1875000000,2.480000e+08,2911000000,5079000000,24520000000,1.220400e+10,NVDA,384000000.0,2.524000e+09


In [17]:
# get necessary market data from yfinance:
# 1. closing price at quarter end
# 2. shares outstanding

import yfinance as yf

t = yf.Ticker("NVDA")

prices = t.history(start="2021-01-01", auto_adjust=True)["Close"]
prices.index = prices.index.tz_localize(None)

shares = t.get_shares_full(start="2021-01-01")
shares.index = shares.index.tz_localize(None)

In [18]:
# map new market data to quarter end dates in df
close_prices= {}
shares_out= {}

for date in df.index:
    p = prices[prices.index <= date]
    close_prices[date] = p.iloc[-1] if not p.empty else np.nan
    
    s = shares[shares.index <= date]
    shares_out[date] = s.iloc[-1] if not s.empty else np.nan

df["close_price"]= pd.Series(close_prices)
df["shares_outstanding"]= pd.Series(shares_out)

In [19]:
# derived market variables we need for EDA and ML:

df["market_cap"]= df["close_price"] * df["shares_outstanding"]
df["free_cash_flow_q"]= df["operating_cf_q"] - df["capex_q"]

print("NaNs:", df[["close_price", "shares_outstanding", "market_cap", "free_cash_flow_q"]].isna().sum())
print(df[["close_price", "shares_outstanding", "market_cap", "free_cash_flow_q"]])

NaNs: close_price           0
shares_outstanding    0
market_cap            0
free_cash_flow_q      0
dtype: int64
             close_price  shares_outstanding    market_cap  free_cash_flow_q
quarter_end                                                                 
2021-05-02     14.954335           622384000  9.307339e+09      1.576000e+09
2021-08-01     19.431786          2492000000  4.842401e+10      2.499000e+09
2021-10-31     25.483360          2492000000  6.350453e+10      1.297000e+09
2022-01-30     22.768074          2492000000  5.673804e+10      2.760000e+09
2022-05-01     18.491741          2492000000  4.608142e+10      1.380667e+09
2022-07-31     18.112717          2492000000  4.513689e+10      9.196667e+08
2022-10-30     13.799800          2492000000  3.438910e+10      4.166667e+07
2023-01-29     20.319849          2492000000  5.063706e+10      1.739000e+09
2023-04-30     27.692181          2470000128  6.839969e+10      2.663000e+09
2023-07-30     46.659103          2470

**Stock split in 2024**  NVIDIA did a 10:1 stock split in June 2024. 

shares_outstanding jumps from 2.5B to 24.5B between 2024-04-28 and 2024-07-28
close_price goes from ~$87 to ~$112 (post-split adjusted)

In [20]:
# final check that we have all basic variables:
print(df.shape)
print(df.isna().sum())

(21, 17)
revenue_q             0
gross_profit_q        0
ebit_q                0
net_income_q          0
rd_expense_q          0
capex_q               0
operating_cf_q        0
cash_q                0
total_equity_q        0
total_debt_q          0
ticker                0
da_q                  0
ebitda_q              0
close_price           0
shares_outstanding    0
market_cap            0
free_cash_flow_q      0
dtype: int64


In [21]:
print(df.head(3).T)

quarter_end                2021-05-02          2021-08-01          2021-10-31
revenue_q                  5661000000          6507000000          7103000000
gross_profit_q             3629000000          4215000000          4631000000
ebit_q                     1956000000          2444000000          2671000000
net_income_q               1912000000          2374000000          2464000000
rd_expense_q               1153000000          1245000000          1403000000
capex_q                   298000000.0         183000000.0         222000000.0
operating_cf_q             1874000000          2682000000          1519000000
cash_q                      978000000          5628000000          1288000000
total_equity_q            18774000000         21147000000         23798000000
total_debt_q             7962000000.0       12943000000.0       10944000000.0
ticker                           NVDA                NVDA                NVDA
da_q                      281000000.0         286000000.0       

In [22]:
# check CIK for the other companies

HEADERS = {"User-Agent": "irenesanch.medina@gmail.com"}

tickers_cik = {
    "AVGO": "Broadcom",
    "MU":"Micron Technology",
    "AMD":"Advanced Micro Devices",
    "INTC":"Intel",
    "TXN":"Texas Instruments",
    "MRVL":"Marvell Technology",
    "QCOM":"Qualcomm",
    "ADI":"Analog Devices",
    "NXPI":"NXP Semiconductors"}

# CIK mapping file
cik_url = "https://www.sec.gov/files/company_tickers.json"
r = requests.get(cik_url, headers=HEADERS)
cik_data = r.json()

# build lookup
cik_lookup = {v["ticker"]: str(v["cik_str"]).zfill(10) 
              for v in cik_data.values()}

for ticker in tickers_cik:
    cik = cik_lookup.get(ticker, "NOT FOUND")
    print(f"{ticker}: {cik}")

AVGO: 0001730168
MU: 0000723125
AMD: 0000002488
INTC: 0000050863
TXN: 0000097476
MRVL: 0001835632
QCOM: 0000804328
ADI: 0000006281
NXPI: 0001413447


# 2. Creating a reusable function

In [23]:
# build reusable function for all tickers
# CIK MAP for other 9 companies
CIK_MAP = { "AVGO": "0001730168",
    "MU":"0000723125",
    "AMD":"0000002488",
    "INTC":"0000050863",
    "TXN":"0000097476",
    "MRVL":"0001835632",
    "QCOM":"0000804328",
    "ADI":"0000006281",
    "NXPI":"0001413447"}

HEADERS = {"User-Agent": "irenesanch.medina@gmail.com"}

# HELPER FUNCTIONS 
def fetch_gaap(cik):
    url  = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    data = requests.get(url, headers=HEADERS).json()
    return data["facts"]["us-gaap"]


def get_balance_sheet_q(gaap, concept, start="2021-01-01"):
    if concept not in gaap:
        return pd.Series(dtype=float, name=concept)
    entries = gaap[concept]["units"]["USD"]
    records = []
    for e in entries:
        if e.get("form") not in ["10-Q", "10-K"]:
            continue
        end = pd.to_datetime(e["end"])
        if end < pd.to_datetime(start):
            continue
        records.append({"quarter_end": end, "value": e["val"]})
    if not records:
        return pd.Series(dtype=float, name=concept)
    df = (pd.DataFrame(records).sort_values("quarter_end").drop_duplicates("quarter_end", keep="last"))
    return df.set_index("quarter_end")["value"]


def get_flow_q(gaap, concept, start="2021-01-01"):
    """extract single-quarter flow values calculated form YTD where needed"""
    if concept not in gaap:
        return pd.Series(dtype=float, name=concept)
    entries = gaap[concept]["units"]["USD"]
    records = []
    for e in entries:
        if e.get("form") not in ["10-Q", "10-K"]:
            continue
        end        = pd.to_datetime(e["end"])
        start_date = pd.to_datetime(e.get("start", pd.NaT))
        if end < pd.to_datetime(start) or pd.isna(start_date):
            continue
        days = (end - start_date).days
        records.append({
            "fp":e.get("fp"),
            "end":end,
            "start":start_date,
            "days":days,
            "val":e["val"]})
    if not records:
        return pd.Series(dtype=float, name=concept)

    df2 = (pd.DataFrame(records).sort_values(["start", "end", "days"]).drop_duplicates(["start", "fp"], keep="last"))

    results = {}
    for fy_start, g in df2.groupby("start"):
        g= g.sort_values("end")
        q1= g[(g["fp"] == "Q1") & g["days"].between(80,  100)]
        q2= g[(g["fp"] == "Q2") & g["days"].between(170, 195)]
        q3= g[(g["fp"] == "Q3") & g["days"].between(260, 285)]
        fy_ = g[(g["fp"] == "FY") & g["days"].between(350, 380)]
        q1_val= q1.iloc[-1]["val"]  if not q1.empty  else None
        q2_val= q2.iloc[-1]["val"]  if not q2.empty  else None
        q3_val= q3.iloc[-1]["val"]  if not q3.empty  else None
        fy_val= fy_.iloc[-1]["val"] if not fy_.empty else None
        q1_end= q1.iloc[-1]["end"]  if not q1.empty  else None
        q2_end= q2.iloc[-1]["end"]  if not q2.empty  else None
        q3_end= q3.iloc[-1]["end"]  if not q3.empty  else None
        fy_end= fy_.iloc[-1]["end"] if not fy_.empty else None
        if q1_val is not None:
            results[q1_end] = q1_val
        if q2_val is not None and q1_val is not None:
            results[q2_end] = q2_val - q1_val
        if q3_val is not None and q2_val is not None:
            results[q3_end] = q3_val - q2_val
        elif q3_val is not None and q1_val is not None:
            results[q3_end] = q3_val - q1_val
        if fy_val is not None and q3_val is not None:
            results[fy_end] = fy_val - q3_val
    return pd.Series(results, name=concept).sort_index()


def get_best_concept(gaap, candidates, mode="flow", start="2021-01-01"):
    """try multiple concept names and return first with enough data"""
    for concept in candidates:
        if concept not in gaap:
            continue
        s = get_flow_q(gaap, concept, start) if mode == "flow" \
            else get_balance_sheet_q(gaap, concept, start)
        if s.notna().sum() >= 4:
            return s
    return pd.Series(dtype=float)


def get_da(gaap, start="2021-01-01"):
    """try all known D&A tags and cmbine if necessary"""
    # try combined tags first
    for concept in ["DepreciationDepletionAndAmortization","DepreciationAndAmortization"]:
        if concept in gaap:
            s = get_flow_q(gaap, concept, start)
            if s.notna().sum() >= 8:
                return s

    # try Depreciation + AmortizationOfIntangibleAssets (AVGO, INTC, TXN, ADI)
    dep = get_flow_q(gaap, "Depreciation", start) \
          if "Depreciation" in gaap else pd.Series(dtype=float)
    ami = get_flow_q(gaap, "AmortizationOfIntangibleAssets", start) \
          if "AmortizationOfIntangibleAssets" in gaap else pd.Series(dtype=float)
    if dep.notna().sum() >= 8:
        return dep.add(ami, fill_value=0)

    # try OtherDepreciationAndAmortization + AmortizationOfIntangibleAssets (AMD, MRVL)
    other_da = get_flow_q(gaap, "OtherDepreciationAndAmortization", start) \
               if "OtherDepreciationAndAmortization" in gaap else pd.Series(dtype=float)
    if other_da.notna().sum() >= 8:
        return other_da.add(ami, fill_value=0)

    return pd.Series(dtype=float)

def get_debt(gaap, start="2021-01-01"):
    """get total debt trying tag combinations"""
    # try combined tags first
    for concept in ["DebtLongtermAndShorttermCombinedAmount",
                "LongTermDebtAndCapitalLeaseObligations",
                "DebtAndCapitalLeaseObligations"]:
        if concept in gaap:
            s = get_balance_sheet_q(gaap, concept, start)
            if s.notna().sum() >= 8:
                # add current portion if available
                curr = get_balance_sheet_q(gaap, "DebtCurrent", start)
                return s.add(curr, fill_value=0)

    # fallback: long term + current separately
    lt = get_best_concept(gaap, [
        "LongTermDebtNoncurrent",
        "LongTermDebt",
        "LongTermDebtAndCapitalLeaseObligations",], mode="bs", start=start)
    st = get_best_concept(gaap, [
        "DebtCurrent",
        "LongTermDebtCurrent",
        "LongTermDebtAndCapitalLeaseObligationsCurrent",], mode="bs", start=start)
    return lt.add(st, fill_value=0)
    
def get_prices_and_shares(ticker, index, start="2021-01-01"):
    """get closing price and shares outstanding from yfinance"""
    t= yf.Ticker(ticker)
    prices = t.history(start=start, auto_adjust=True)["Close"]
    prices.index = prices.index.tz_localize(None)
    shares = t.get_shares_full(start=start)
    shares.index = shares.index.tz_localize(None)

    close, sh = {}, {}
    for date in index:
        p = prices[prices.index <= date]
        s = shares[shares.index <= date]
        close[date]= p.iloc[-1] if not p.empty else np.nan
        sh[date]= s.iloc[-1] if not s.empty else np.nan
    return pd.Series(close, name="close_price"), pd.Series(sh, name="shares_outstanding")


In [24]:
# MAIN FUNCTION
def build_company_df(ticker, start="2021-01-01"):
    print(f"\nFetching {ticker}...")
    cik  = CIK_MAP[ticker]
    gaap = fetch_gaap(cik)

    df = pd.DataFrame()

    # INCOME STATEMENT
    df["revenue_q"] = get_best_concept(gaap, ["RevenueFromContractWithCustomerExcludingAssessedTax","Revenues", "SalesRevenueNet"], mode="flow", start=start)
    df["gross_profit_q"] = get_best_concept(gaap, ["GrossProfit"], mode="flow", start=start)
    # derive if missing (QCOM case — no GrossProfit tag)
    if df["gross_profit_q"].isna().sum() > 10:
        cost_of_rev = get_best_concept(gaap, ["CostOfRevenue","CostOfGoodsAndServicesSold","CostOfGoodsSold"], mode="flow", start=start)
        df["gross_profit_q"] = df["revenue_q"] - cost_of_rev
    df["ebit_q"]= get_flow_q(gaap, "OperatingIncomeLoss", start)
    df["net_income_q"] = get_best_concept(gaap, ["NetIncomeLoss","ProfitLoss", "NetIncomeLossAvailableToCommonStockholdersBasic",], mode="flow", start=start)    
    df["rd_expense_q"]= get_flow_q(gaap, "ResearchAndDevelopmentExpense", start)

    # CASH FLOW
    df["operating_cf_q"]= get_flow_q(gaap, "NetCashProvidedByUsedInOperatingActivities", start)
    df["capex_q"]= get_best_concept(gaap, [
        "PaymentsToAcquireProductiveAssets",
        "PaymentsToAcquirePropertyPlantAndEquipment",
        "PaymentsForCapitalImprovements"], mode="flow", start=start)

    # BALANCE SHEET
    df["cash_q"] = get_best_concept(gaap, ["CashAndCashEquivalentsAtCarryingValue","CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents", "CashCashEquivalentsAndShortTermInvestments",], mode="bs", start=start)
    df["total_equity_q"] = get_best_concept(gaap, ["StockholdersEquity","CommonStockEquity","StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"], mode="bs", start=start)
    df["total_debt_q"] = get_debt(gaap, start)

    # D&A + EBITDA
    df["da_q"]= get_da(gaap, start)
    df["ebitda_q"]= df["ebit_q"] + df["da_q"]

    # CLEAN INDEX 
    df = df[df.index.notna()].sort_index()
    df.index.name = "quarter_end"

    # PRICE + SHARES FROM YFINANCE
    close, shares= get_prices_and_shares(ticker, df.index, start)
    df["close_price"]= close
    df["shares_outstanding"]= shares
    df["market_cap"]= df["close_price"] * df["shares_outstanding"]
    df["free_cash_flow_q"]= df["operating_cf_q"] - df["capex_q"]
    df["ticker"]= ticker

    print(f"  {ticker}: {len(df)} quarters, {df.isna().sum().sum()} total NaNs")
    print(f"     NaNs per column: {df.isna().sum().to_dict()}")

    return df

# 3. Other 9 companies

In [25]:
# RUN FOR ALL 9 TICKERS
all_dfs = []
for ticker in CIK_MAP:
    df_ticker = build_company_df(ticker)
    all_dfs.append(df_ticker)

master_df = pd.concat(all_dfs).sort_values(["ticker", "quarter_end"])
print(f"\n Master df shape: {master_df.shape}")
print(master_df.groupby("ticker").size())


Fetching AVGO...
  AVGO: 22 quarters, 14 total NaNs
     NaNs per column: {'revenue_q': 0, 'gross_profit_q': 0, 'ebit_q': 0, 'net_income_q': 0, 'rd_expense_q': 0, 'operating_cf_q': 0, 'capex_q': 0, 'cash_q': 0, 'total_equity_q': 0, 'total_debt_q': 12, 'da_q': 0, 'ebitda_q': 0, 'close_price': 0, 'shares_outstanding': 1, 'market_cap': 1, 'free_cash_flow_q': 0, 'ticker': 0}

Fetching MU...
  MU: 21 quarters, 0 total NaNs
     NaNs per column: {'revenue_q': 0, 'gross_profit_q': 0, 'ebit_q': 0, 'net_income_q': 0, 'rd_expense_q': 0, 'operating_cf_q': 0, 'capex_q': 0, 'cash_q': 0, 'total_equity_q': 0, 'total_debt_q': 0, 'da_q': 0, 'ebitda_q': 0, 'close_price': 0, 'shares_outstanding': 0, 'market_cap': 0, 'free_cash_flow_q': 0, 'ticker': 0}

Fetching AMD...
  AMD: 21 quarters, 3 total NaNs
     NaNs per column: {'revenue_q': 0, 'gross_profit_q': 0, 'ebit_q': 0, 'net_income_q': 0, 'rd_expense_q': 0, 'operating_cf_q': 0, 'capex_q': 0, 'cash_q': 0, 'total_equity_q': 0, 'total_debt_q': 3, 'da_q':

In [26]:
# interpolate remaining NaNs in total_debt_q cash_q per company
for ticker in master_df["ticker"].unique():
    mask = master_df["ticker"] == ticker
    for col in ["total_debt_q", "cash_q", "net_income_q"]:
        master_df.loc[mask, col] = (master_df.loc[mask, col].interpolate(method="linear", limit_direction="both"))

print("Total NaNs after interpolation:")
print(master_df.isna().sum())
print("\nTotal NaNs:", master_df.isna().sum().sum())

Total NaNs after interpolation:
revenue_q             0
gross_profit_q        0
ebit_q                0
net_income_q          0
rd_expense_q          0
operating_cf_q        0
capex_q               0
cash_q                0
total_equity_q        0
total_debt_q          0
da_q                  0
ebitda_q              0
close_price           0
shares_outstanding    1
market_cap            1
free_cash_flow_q      0
ticker                0
dtype: int64

Total NaNs: 2


In [27]:
missing_rows = master_df[master_df[["shares_outstanding", "market_cap"]].isna().any(axis=1)]
missing_rows

,revenue_q,gross_profit_q,ebit_q,net_income_q,rd_expense_q,operating_cf_q,capex_q,cash_q,total_equity_q,total_debt_q,da_q,ebitda_q,close_price,shares_outstanding,market_cap,free_cash_flow_q,ticker
quarter_end,,,,,,,,,,,,,,,,,
2021-01-31,6655000000,3952000000,1837000000,1378000000,1211000000,3113000000,114000000,9.552000e+09,23973000000,4.242600e+10,138000000.0,1.975000e+09,40.378578,NaN,NaN,2999000000,AVGO


In [28]:
nvda_df = df.copy()

# make sure quarter_end is an index 
if "quarter_end" in nvda_df.columns:
    nvda_df["quarter_end"] = pd.to_datetime(nvda_df["quarter_end"])
    nvda_df = nvda_df.set_index("quarter_end")
else:
    nvda_df.index = pd.to_datetime(nvda_df.index)
    nvda_df.index.name = "quarter_end"

nvda_df["ticker"] = "NVDA"

# free cash flow exists
nvda_df["free_cash_flow_q"] = (nvda_df["operating_cf_q"] - nvda_df["capex_q"])

# market cap 
nvda_df["market_cap"] = (nvda_df["close_price"] * nvda_df["shares_outstanding"])

In [29]:
nvda_df.head(20)

,revenue_q,gross_profit_q,ebit_q,net_income_q,rd_expense_q,capex_q,operating_cf_q,cash_q,total_equity_q,total_debt_q,ticker,da_q,ebitda_q,close_price,shares_outstanding,market_cap,free_cash_flow_q
quarter_end,,,,,,,,,,,,,,,,,
2021-05-02,5661000000,3629000000,1956000000,1912000000,1153000000,2.980000e+08,1874000000,978000000,18774000000,7.962000e+09,NVDA,281000000.0,2.237000e+09,14.954335,622384000,9.307339e+09,1.576000e+09
2021-08-01,6507000000,4215000000,2444000000,2374000000,1245000000,1.830000e+08,2682000000,5628000000,21147000000,1.294300e+10,NVDA,286000000.0,2.730000e+09,19.431786,2492000000,4.842401e+10,2.499000e+09
2021-10-31,7103000000,4631000000,2671000000,2464000000,1403000000,2.220000e+08,1519000000,1288000000,23798000000,1.094400e+10,NVDA,297500000.0,2.968500e+09,25.483360,2492000000,6.350453e+10,1.297000e+09
2022-01-30,7643000000,4999000000,2970000000,3003000000,1466000000,2.730000e+08,3033000000,1990000000,26612000000,1.094600e+10,NVDA,309000000.0,3.279000e+09,22.768074,2492000000,5.673804e+10,2.760000e+09
2022-05-01,8288000000,5431000000,1868000000,1618000000,1618000000,3.503333e+08,1731000000,3887000000,26320000000,1.094700e+10,NVDA,334000000.0,2.202000e+09,18.491741,2492000000,4.608142e+10,1.380667e+09
2022-07-31,6704000000,2915000000,499000000,656000000,1824000000,3.503333e+08,1270000000,3013000000,23851000000,1.219800e+10,NVDA,378000000.0,8.770000e+08,18.112717,2492000000,4.513689e+10,9.196667e+08
2022-10-30,5931000000,3177000000,601000000,680000000,1945000000,3.503333e+08,392000000,2800000000,21349000000,1.219900e+10,NVDA,406000000.0,1.007000e+09,13.799800,2492000000,3.438910e+10,4.166667e+07
2023-01-29,6051000000,3833000000,1256000000,1414000000,1952000000,5.090000e+08,2248000000,3389000000,22101000000,1.220300e+10,NVDA,426000000.0,1.682000e+09,20.319849,2492000000,5.063706e+10,1.739000e+09
2023-04-30,7192000000,4648000000,2140000000,2043000000,1875000000,2.480000e+08,2911000000,5079000000,24520000000,1.220400e+10,NVDA,384000000.0,2.524000e+09,27.692181,2470000128,6.839969e+10,2.663000e+09


In [30]:
# add any columns missing from NVDA
for col in master_df.columns:
    if col not in nvda_df.columns:
        nvda_df[col] = pd.NA

# add any columns missing from master_df, just in case
for col in nvda_df.columns:
    if col not in master_df.columns:
        master_df[col] = pd.NA

# reorder NVDA columns to match master_df
nvda_df = nvda_df[master_df.columns]

In [31]:
print(nvda_df.shape)
print(master_df.shape)

(21, 17)
(190, 17)


In [32]:
# join NVDA data to master_df
master_df_all = (pd.concat([master_df, nvda_df]).sort_values(["ticker", "quarter_end"]))

print(master_df_all.index.name)
print(master_df_all.shape)
print(master_df_all.groupby("ticker").size())
print(master_df_all.isna().sum())

quarter_end
(211, 17)
ticker
ADI     22
AMD     21
AVGO    22
INTC    21
MRVL    21
MU      21
NVDA    21
NXPI    21
QCOM    20
TXN     21
dtype: int64
revenue_q             0
gross_profit_q        0
ebit_q                0
net_income_q          0
rd_expense_q          0
operating_cf_q        0
capex_q               0
cash_q                0
total_equity_q        0
total_debt_q          0
da_q                  0
ebitda_q              0
close_price           0
shares_outstanding    1
market_cap            1
free_cash_flow_q      0
ticker                0
dtype: int64


# 4. Derived ratios and markers

In [33]:
# calculate TTM values for relevant variables

master_df = master_df_all.copy()
master_df = master_df.sort_values("quarter_end")

def ttm_by_ticker(df, col):
    return (
        df.groupby("ticker")[col]
        .transform(lambda s: s.rolling(4, min_periods=4).sum()))

master_df["revenue_ttm"] = ttm_by_ticker(master_df, "revenue_q")
master_df["gross_profit_ttm"] = ttm_by_ticker(master_df, "gross_profit_q")
master_df["ebit_ttm"] = ttm_by_ticker(master_df, "ebit_q")
master_df["net_income_ttm"] = ttm_by_ticker(master_df, "net_income_q")
master_df["rd_ttm"] = ttm_by_ticker(master_df, "rd_expense_q")
master_df["capex_ttm"] = ttm_by_ticker(master_df, "capex_q")
master_df["operating_cf_ttm"] = ttm_by_ticker(master_df, "operating_cf_q")
master_df["free_cash_flow_q"] = (master_df["operating_cf_q"] - master_df["capex_q"])
master_df["free_cash_flow_ttm"] = ttm_by_ticker(master_df, "free_cash_flow_q")
master_df["ebitda_ttm"] = ttm_by_ticker(master_df, "ebitda_q")

In [34]:
display(master_df.isna().sum())
display(master_df.columns)

revenue_q              0
gross_profit_q         0
ebit_q                 0
net_income_q           0
rd_expense_q           0
operating_cf_q         0
capex_q                0
cash_q                 0
total_equity_q         0
total_debt_q           0
da_q                   0
ebitda_q               0
close_price            0
shares_outstanding     1
market_cap             1
free_cash_flow_q       0
ticker                 0
revenue_ttm           30
gross_profit_ttm      30
ebit_ttm              30
net_income_ttm        30
rd_ttm                30
capex_ttm             30
operating_cf_ttm      30
free_cash_flow_ttm    30
ebitda_ttm            30
dtype: int64

Index(['revenue_q', 'gross_profit_q', 'ebit_q', 'net_income_q', 'rd_expense_q',
       'operating_cf_q', 'capex_q', 'cash_q', 'total_equity_q', 'total_debt_q',
       'da_q', 'ebitda_q', 'close_price', 'shares_outstanding', 'market_cap',
       'free_cash_flow_q', 'ticker', 'revenue_ttm', 'gross_profit_ttm',
       'ebit_ttm', 'net_income_ttm', 'rd_ttm', 'capex_ttm', 'operating_cf_ttm',
       'free_cash_flow_ttm', 'ebitda_ttm'],
      dtype='str')

In [35]:
# profitability variables:
master_df["gross_margin"] = master_df["gross_profit_ttm"] / master_df["revenue_ttm"]
master_df["operating_margin"] = master_df["ebit_ttm"] / master_df["revenue_ttm"]
master_df["ebitda_margin"] = master_df["ebitda_ttm"] / master_df["revenue_ttm"]
master_df["net_margin"] = master_df["net_income_ttm"] / master_df["revenue_ttm"]
master_df["avg_equity_ttm"] = (master_df.groupby("ticker")["total_equity_q"].transform(lambda s: s.rolling(4, min_periods=4).mean()))
master_df["roe"] = master_df["net_income_ttm"] / master_df["avg_equity_ttm"]

In [36]:
# semi-conductor specififc variables:
master_df["rd_to_revenue"] = master_df["rd_ttm"] / master_df["revenue_ttm"]
master_df["capex_to_revenue"] = master_df["capex_ttm"] / master_df["revenue_ttm"]
master_df["debt_to_ebitda"] = master_df["total_debt_q"] / master_df["ebitda_ttm"]

In [37]:
# growth variables (quarterly yoy growth=same quarter vs same quarter prev year)
master_df["revenue_growth_yoy"] = (master_df.groupby("ticker")["revenue_q"].pct_change(4))
master_df["ebitda_growth_yoy"] = (master_df.groupby("ticker")["ebitda_q"].pct_change(4))
master_df["net_income_growth_yoy"] = (master_df.groupby("ticker")["net_income_q"].pct_change(4))

In [38]:
display(master_df.isna().sum())
display(master_df.columns)

revenue_q                 0
gross_profit_q            0
ebit_q                    0
net_income_q              0
rd_expense_q              0
operating_cf_q            0
capex_q                   0
cash_q                    0
total_equity_q            0
total_debt_q              0
da_q                      0
ebitda_q                  0
close_price               0
shares_outstanding        1
market_cap                1
free_cash_flow_q          0
ticker                    0
revenue_ttm              30
gross_profit_ttm         30
ebit_ttm                 30
net_income_ttm           30
rd_ttm                   30
capex_ttm                30
operating_cf_ttm         30
free_cash_flow_ttm       30
ebitda_ttm               30
gross_margin             30
operating_margin         30
ebitda_margin            30
net_margin               30
avg_equity_ttm           30
roe                      30
rd_to_revenue            30
capex_to_revenue         30
debt_to_ebitda           30
revenue_growth_yoy  

Index(['revenue_q', 'gross_profit_q', 'ebit_q', 'net_income_q', 'rd_expense_q',
       'operating_cf_q', 'capex_q', 'cash_q', 'total_equity_q', 'total_debt_q',
       'da_q', 'ebitda_q', 'close_price', 'shares_outstanding', 'market_cap',
       'free_cash_flow_q', 'ticker', 'revenue_ttm', 'gross_profit_ttm',
       'ebit_ttm', 'net_income_ttm', 'rd_ttm', 'capex_ttm', 'operating_cf_ttm',
       'free_cash_flow_ttm', 'ebitda_ttm', 'gross_margin', 'operating_margin',
       'ebitda_margin', 'net_margin', 'avg_equity_ttm', 'roe', 'rd_to_revenue',
       'capex_to_revenue', 'debt_to_ebitda', 'revenue_growth_yoy',
       'ebitda_growth_yoy', 'net_income_growth_yoy'],
      dtype='str')

In [39]:
# EV -> Market Cap + Total Debt - Cash
# enterprise value
master_df["enterprise_value"] = (master_df["market_cap"] + master_df["total_debt_q"] - master_df["cash_q"])

# valuation multiples
master_df["pe_ratio"] = master_df["market_cap"] / master_df["net_income_ttm"]
master_df["ev_to_ebitda"] = master_df["enterprise_value"] / master_df["ebitda_ttm"]
master_df["price_to_sales"] = master_df["market_cap"] / master_df["revenue_ttm"]

In [40]:
# add company names for easier EDA
COMPANY_MAP = {
    "NVDA": "NVIDIA Corporation",
    "AVGO": "Broadcom Inc.",
    "MU": "Micron Technology Inc.",
    "AMD": "Advanced Micro Devices Inc.",
    "INTC": "Intel Corporation",
    "TXN": "Texas Instruments Incorporated",
    "MRVL": "Marvell Technology Inc.",
    "QCOM": "Qualcomm Incorporated",
    "ADI": "Analog Devices Inc.",
    "NXPI": "NXP Semiconductors N.V.",}

master_df["company"] = master_df["ticker"].map(COMPANY_MAP)

In [41]:
# clean quarter label
master_df = master_df.sort_values(["ticker", "quarter_end"])

master_df["quarter_num"] = (master_df.groupby("ticker").cumcount().add(1))

master_df["fiscal_quarter_num"] = (master_df.groupby("ticker").cumcount().mod(4).add(1))

master_df["fiscal_year_num"] = (master_df.groupby("ticker").cumcount().floordiv(4).add(2021))

master_df["fiscal_quarter_label"] = (master_df["fiscal_year_num"].astype(str)+ "Q"+ master_df["fiscal_quarter_num"].astype(str))

In [42]:
display(master_df.quarter_num.value_counts())
display(master_df.fiscal_quarter_num.value_counts())
display(master_df.fiscal_year_num.value_counts())
display(master_df.fiscal_quarter_label.value_counts())

quarter_num
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11    10
12    10
13    10
14    10
15    10
16    10
17    10
18    10
19    10
20    10
21     9
22     2
Name: count, dtype: int64

fiscal_quarter_num
1    59
2    52
3    50
4    50
Name: count, dtype: int64

fiscal_year_num
2021    40
2022    40
2023    40
2024    40
2025    40
2026    11
Name: count, dtype: int64

fiscal_quarter_label
2021Q1    10
2021Q2    10
2021Q3    10
2021Q4    10
2022Q1    10
2022Q2    10
2022Q3    10
2022Q4    10
2023Q1    10
2023Q2    10
2023Q3    10
2023Q4    10
2024Q1    10
2024Q2    10
2024Q3    10
2024Q4    10
2025Q1    10
2025Q2    10
2025Q3    10
2025Q4    10
2026Q1     9
2026Q2     2
Name: count, dtype: int64

In [43]:
master_df = master_df.reset_index()

master_df["quarter_end"] = pd.to_datetime(master_df["quarter_end"])

clean_df = (master_df[master_df["quarter_end"] >= "2022-01-01"].sort_values(["ticker", "quarter_end"]).copy())

clean_df.head()

,quarter_end,revenue_q,gross_profit_q,ebit_q,net_income_q,rd_expense_q,operating_cf_q,capex_q,cash_q,total_equity_q,...,net_income_growth_yoy,enterprise_value,pe_ratio,ev_to_ebitda,price_to_sales,company,quarter_num,fiscal_quarter_num,fiscal_year_num,fiscal_quarter_label
4,2022-01-29,2684293000,1401997000,364757000,280077000,426780000,856413000,111133000.0,1.790399e+09,37427312000,...,-0.279116,7.539321e+10,60.206564,24.831528,9.140515,Analog Devices Inc.,5,1,2022,2022Q1
5,2022-04-30,2972064000,1944520000,918161000,783273000,420901000,1221807000,118779000.0,1.737733e+09,37099782000,...,0.852125,7.345487e+10,45.783600,19.266351,7.708284,Analog Devices Inc.,6,2,2022,2022Q2
6,2022-07-30,3109880000,2043142000,893306000,748985000,431829000,1247846000,164884000.0,1.524960e+09,36638591000,...,0.488116,8.455400e+10,45.592140,18.877551,7.750808,Analog Devices Inc.,7,3,2022,2022Q3
7,2022-10-29,3247716000,2142815000,1102476000,936226000,421008000,1149336000,304512000.0,1.470572e+09,36465323000,...,11.369707,7.509189e+10,25.472908,13.466502,5.827711,Analog Devices Inc.,8,4,2022,2022Q4
8,2023-01-28,3249630000,2124341000,1130820000,961474000,414095000,1406305000,176158000.0,1.670462e+09,36531485000,...,2.432892,8.635316e+10,23.801031,13.577647,6.489757,Analog Devices Inc.,9,1,2023,2023Q1


In [44]:
print(clean_df.shape)
print(clean_df.groupby("ticker").size())
print(clean_df.isna().sum())

(175, 48)
ticker
ADI     18
AMD     17
AVGO    18
INTC    17
MRVL    18
MU      18
NVDA    18
NXPI    17
QCOM    17
TXN     17
dtype: int64
quarter_end              0
revenue_q                0
gross_profit_q           0
ebit_q                   0
net_income_q             0
rd_expense_q             0
operating_cf_q           0
capex_q                  0
cash_q                   0
total_equity_q           0
total_debt_q             0
da_q                     0
ebitda_q                 0
close_price              0
shares_outstanding       0
market_cap               0
free_cash_flow_q         0
ticker                   0
revenue_ttm              0
gross_profit_ttm         0
ebit_ttm                 0
net_income_ttm           0
rd_ttm                   0
capex_ttm                0
operating_cf_ttm         0
free_cash_flow_ttm       0
ebitda_ttm               0
gross_margin             0
operating_margin         0
ebitda_margin            0
net_margin               0
avg_equity_ttm         

In [45]:
# dropping unnecesary columns and fixing their order: 

clean_df = clean_df.drop(columns=[ "quarter","quarter_num","fiscal_quarter_num","fiscal_year_num",],errors="ignore")
clean_df = clean_df.drop(columns=["level_0", "index"],errors="ignore")

id_cols = ["ticker", "company", "quarter_end", "fiscal_quarter_label"]
other_cols = [col for col in clean_df.columns if col not in id_cols]

clean_df = clean_df[id_cols + other_cols]

In [46]:
print(clean_df.shape)
print(clean_df.columns)
print(clean_df.isna().sum())

(175, 45)
Index(['ticker', 'company', 'quarter_end', 'fiscal_quarter_label', 'revenue_q',
       'gross_profit_q', 'ebit_q', 'net_income_q', 'rd_expense_q',
       'operating_cf_q', 'capex_q', 'cash_q', 'total_equity_q', 'total_debt_q',
       'da_q', 'ebitda_q', 'close_price', 'shares_outstanding', 'market_cap',
       'free_cash_flow_q', 'revenue_ttm', 'gross_profit_ttm', 'ebit_ttm',
       'net_income_ttm', 'rd_ttm', 'capex_ttm', 'operating_cf_ttm',
       'free_cash_flow_ttm', 'ebitda_ttm', 'gross_margin', 'operating_margin',
       'ebitda_margin', 'net_margin', 'avg_equity_ttm', 'roe', 'rd_to_revenue',
       'capex_to_revenue', 'debt_to_ebitda', 'revenue_growth_yoy',
       'ebitda_growth_yoy', 'net_income_growth_yoy', 'enterprise_value',
       'pe_ratio', 'ev_to_ebitda', 'price_to_sales'],
      dtype='str')
ticker                   0
company                  0
quarter_end              0
fiscal_quarter_label     0
revenue_q                0
gross_profit_q           0
ebit_q   

In [47]:
print(cfg)
print(cfg.keys())

{'raw': {'semiconductor_quarterly': '../data/raw/semiconductor_quarterly.csv'}, 'clean': {'semiconductor_quarterly': '../data/clean/semiconductor_quarterly_clean.csv', 'feature_engineering': '../data/clean/feature_selection.csv', 'ml1_results': '../data/clean/ml1_results.csv'}, 'outputs': {'figures': '../data/graphs'}}
dict_keys(['raw', 'clean', 'outputs'])


In [48]:
# save df to the clean data defined in config:
clean_df.to_csv(cfg['clean']['semiconductor_quarterly'], index=False)

print(f"File saved to: {cfg['clean']['semiconductor_quarterly']}")

File saved to: ../data/clean/semiconductor_quarterly_clean.csv
